In [5]:
# 02_cleaning_feature_engineering.ipynb
# Step 1: imports and load raw matches

import os
import pandas as pd
import numpy as np

# assume notebook is in football_capstone/notebooks
base_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(base_dir, "data")

raw_file = os.path.join(data_dir, "raw_epl_matches_2seasons.csv")
print("Raw file:", raw_file)

df_matches = pd.read_csv(raw_file)
print("Match-level shape:", df_matches.shape)
df_matches.head()


Raw file: C:\Users\Mohsin\football_capstone\data\raw_epl_matches_2seasons.csv
Match-level shape: (760, 106)


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA
0,E0,05/08/2022,20:00,Crystal Palace,Arsenal,0,2,A,0,1,...,1.76,0.50,2.09,1.84,2.04,1.88,2.09,1.88,2.03,1.85
1,E0,06/08/2022,12:30,Fulham,Liverpool,2,2,D,1,0,...,2.73,1.75,1.90,2.03,1.91,2.02,2.01,2.06,1.89,1.99
2,E0,06/08/2022,15:00,Bournemouth,Aston Villa,2,0,H,1,0,...,1.76,0.50,1.93,2.00,1.93,2.00,1.94,2.04,1.88,2.00
3,E0,06/08/2022,15:00,Leeds,Wolves,2,1,H,1,1,...,1.87,-0.25,2.08,1.85,2.10,1.84,2.14,1.87,2.08,1.81
4,E0,06/08/2022,15:00,Newcastle,Nott'm Forest,2,0,H,0,0,...,1.89,-1.00,1.97,1.96,1.99,1.93,2.19,1.97,2.03,1.86


In [6]:
# Step 2: keep only key columns and rename them

cols_needed = [
    "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG",          # full-time goals
    "HS", "AS",              # shots
    "HST", "AST",            # shots on target
    "HC", "AC",              # corners
]

df = df_matches[cols_needed].copy()

df.rename(columns={
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_sot",
    "AST": "away_sot",
    "HC": "home_corners",
    "AC": "away_corners",
}, inplace=True)

print("After column selection:", df.shape)
df.head()


After column selection: (760, 11)


,Date,home_team,away_team,home_goals,away_goals,home_shots,away_shots,home_sot,away_sot,home_corners,away_corners
0,05/08/2022,Crystal Palace,Arsenal,0,2,10,10,2,2,3,5
1,06/08/2022,Fulham,Liverpool,2,2,9,11,3,4,4,4
2,06/08/2022,Bournemouth,Aston Villa,2,0,7,15,3,2,5,5
3,06/08/2022,Leeds,Wolves,2,1,12,15,4,6,6,4
4,06/08/2022,Newcastle,Nott'm Forest,2,0,23,5,10,0,11,1


In [7]:
# Step 3: basic cleaning and derived match fields

# Convert Date to datetime (day first)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

# Drop rows with missing date or goals
df = df.dropna(subset=["Date", "home_goals", "away_goals"])

# Create total goals, result, and over 2.5 goals flag
df["total_goals"] = df["home_goals"] + df["away_goals"]

df["result"] = np.where(
    df["home_goals"] > df["away_goals"], "H",
    np.where(df["home_goals"] < df["away_goals"], "A", "D")
)

df["over25"] = (df["total_goals"] > 2.5).astype(int)

print("Cleaned match-level shape:", df.shape)
print(df.columns.tolist())
df.head()


Cleaned match-level shape: (760, 14)
['Date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_shots', 'away_shots', 'home_sot', 'away_sot', 'home_corners', 'away_corners', 'total_goals', 'result', 'over25']


,Date,home_team,away_team,home_goals,away_goals,home_shots,away_shots,home_sot,away_sot,home_corners,away_corners,total_goals,result,over25
0,2022-08-05,Crystal Palace,Arsenal,0,2,10,10,2,2,3,5,2,A,0
1,2022-08-06,Fulham,Liverpool,2,2,9,11,3,4,4,4,4,D,1
2,2022-08-06,Bournemouth,Aston Villa,2,0,7,15,3,2,5,5,2,H,0
3,2022-08-06,Leeds,Wolves,2,1,12,15,4,6,6,4,3,H,1
4,2022-08-06,Newcastle,Nott'm Forest,2,0,23,5,10,0,11,1,2,H,0


In [8]:
# Step 4: convert match-level data to team-match level (home row + away row)

home_cols = {
    "Date": "Date",
    "home_team": "team",
    "away_team": "opponent",
    "home_goals": "goals_for",
    "away_goals": "goals_against",
    "home_shots": "shots_for",
    "away_shots": "shots_against",
    "home_sot": "sot_for",
    "away_sot": "sot_against",
    "home_corners": "corners_for",
    "away_corners": "corners_against",
    "total_goals": "total_goals",
    "over25": "over25",
}

away_cols = {
    "Date": "Date",
    "away_team": "team",
    "home_team": "opponent",
    "away_goals": "goals_for",
    "home_goals": "goals_against",
    "away_shots": "shots_for",
    "home_shots": "shots_against",
    "away_sot": "sot_for",
    "home_sot": "sot_against",
    "away_corners": "corners_for",
    "home_corners": "corners_against",
    "total_goals": "total_goals",
    "over25": "over25",
}

df_home = df[list(home_cols.keys())].rename(columns=home_cols)
df_home["is_home"] = 1

df_away = df[list(away_cols.keys())].rename(columns=away_cols)
df_away["is_home"] = 0

df_team = pd.concat([df_home, df_away], ignore_index=True)
print("Team-match shape:", df_team.shape)
df_team.head()


Team-match shape: (1520, 14)


,Date,team,opponent,goals_for,goals_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,total_goals,over25,is_home
0,2022-08-05,Crystal Palace,Arsenal,0,2,10,10,2,2,3,5,2,0,1
1,2022-08-06,Fulham,Liverpool,2,2,9,11,3,4,4,4,4,1,1
2,2022-08-06,Bournemouth,Aston Villa,2,0,7,15,3,2,5,5,2,0,1
3,2022-08-06,Leeds,Wolves,2,1,12,15,4,6,6,4,3,1,1
4,2022-08-06,Newcastle,Nott'm Forest,2,0,23,5,10,0,11,1,2,0,1


In [9]:
# Step 5: add a few simple engineered features

df_team["goal_diff"] = df_team["goals_for"] - df_team["goals_against"]
df_team["is_win"] = (df_team["goal_diff"] > 0).astype(int)
df_team["is_draw"] = (df_team["goal_diff"] == 0).astype(int)
df_team["is_loss"] = (df_team["goal_diff"] < 0).astype(int)

# Sort by team and date (useful later)
df_team = df_team.sort_values(["team", "Date"]).reset_index(drop=True)

print("Final team-match shape:", df_team.shape)
df_team.head()


Final team-match shape: (1520, 18)


,Date,team,opponent,goals_for,goals_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,total_goals,over25,is_home,goal_diff,is_win,is_draw,is_loss
0,2022-08-05,Arsenal,Crystal Palace,2,0,10,10,2,2,5,3,2,0,0,2,1,0,0
1,2022-08-13,Arsenal,Leicester,4,2,19,6,7,2,6,2,6,1,1,2,1,0,0
2,2022-08-20,Arsenal,Bournemouth,3,0,14,6,6,1,4,3,3,1,0,3,1,0,0
3,2022-08-27,Arsenal,Fulham,2,1,22,11,8,3,9,2,3,1,1,1,1,0,0
4,2022-08-31,Arsenal,Aston Villa,2,1,22,4,9,3,10,3,3,1,1,1,1,0,0


In [10]:
# Step 6: save clean team-match dataset

clean_path = os.path.join(data_dir, "epl_team_matches_clean.csv")
df_team.to_csv(clean_path, index=False)

print("Saved clean team-match data to:", clean_path)


Saved clean team-match data to: C:\Users\Mohsin\football_capstone\data\epl_team_matches_clean.csv
